# Test ASR sur KALLAAMA — Wolof

> **Objectif** — Établir une baseline de reconnaissance de la parole wolof
> sur le corpus caractérisé au notebook 1. Diagnostiquer cinq modèles
> (Whisper vanilla, M9and2M, MMS, Whosper-large, Wolof-HuBERT), mesurer
> leur performance (WER et CER) sur les segments évaluables et sur un
> corpus de contrôle (FLEURS), et quantifier leur comportement sur le
> code-switching wolof–français — pour décider quel modèle porter en
> fine-tuning et sur quelles bases.
>
> **Livrables**
> - `notebooks/02_asr_baseline.ipynb` — ce notebook
> - `normalize_for_wer`, `nettoyer_speaker` (dans `src/kallaama.py`) —
>   normalisation symétrique référence/hypothèse et correction du champ
>   speaker
> - un tableau de baseline : WER et CER par modèle, sur segments courts,
>   moyens, longs, un échantillon élargi, et FLEURS
---
## Le corpus

**KALLAAMA** — *A Transcribed Speech Dataset about Agriculture in the Three Most
Widely Spoken Languages in Senegal* (production 2023, publication RAIL 2024).

Thématique **agriculture** : alignement direct avec Ñoo Far.

| | |
|---|---|
| **Porteur** | Jokalante (Dakar) |
| **Partenaires** | Orange Innovation (Lannion), École Polytechnique de Thiès |
| **Financement** | Lacuna Fund |
| **Langues** | wolof (`wol`), pulaar (`fuc`), sereer (`srr`) |
| **Audio** | WAV, 16 kHz, 16 bits, mono |
| **Transcriptions** | `.stm` (NIST) — **format retenu** · `.trs` (Transcriber XML) — métadonnées locuteur |
| **Licence** | CC-BY 4.0 — usage commercial, modification et redistribution autorisés, sous réserve d'attribution |

### Volumes — wolof

| Set | Fichiers | Audio | Parole annotée |
|---|---:|---:|---:|
| Whole | 153 | 55 h 11 | 51 h 08 |
| **Checked** | **36** | **12 h 49** | **11 h 47** |

*Chiffres annoncés par les auteurs.* La « parole annotée » recouvre ici
l'ensemble des segments transcrits, y compris les intervalles exclus du
scoring et les tours non-verbaux. La volumétrie réellement exploitable —
après filtrage — est établie au **notebook 1** : sur 13 061 lignes,
11 098 segments de parole, dont 6 960 directement évaluables.

Le *checked set* est un sous-ensemble vérifié manuellement du *whole set* :
mêmes audios, transcriptions relues. C'est lui qui sert de **corpus
d'évaluation** pour le baseline WER.

## Section 0 — Setup
Chargement du corpus depuis `corpus_clean.csv` (produit par le notebook 1),
construction de `audio_map`, import du module.

In [ ]:
from pathlib import Path
import sys

# Environnement détecté automatiquement Colab/Local
try:
    import google.colab
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path("/content/drive/MyDrive/noo-far-pipeline")

    # Dépendances non préinstallées sur Colab, ou versions trop anciennes
    !pip install -U torchao --quiet    # requis par peft pour whosper-large (LoRA)
    !pip install jiwer --quiet          # calcul WER/CER, absent par défaut
else:
    PROJECT_ROOT = Path(r"C:\dev\noo-far-pipeline")

DATA_DIR   = PROJECT_ROOT / "data"
CORPUS_DIR = DATA_DIR / "kallaama" / "wolof" / "speech_dataset"
CSV        = DATA_DIR / "corpus_clean.csv"

sys.path.insert(0, str(PROJECT_ROOT / "src"))

print("Environnement :", "Colab" if ON_COLAB else "Local")
print("PROJECT_ROOT  :", PROJECT_ROOT)

In [ ]:
import gc
import pandas as pd
import numpy as np
import librosa, librosa.display
import matplotlib.pyplot as plt
import torch
from transformers import (
    WhisperProcessor, WhisperForConditionalGeneration, WhisperTokenizer,
    AutoProcessor, AutoModelForCTC, Wav2Vec2ForCTC,
)

import jiwer

from kallaama import extract_segment, normalize_for_wer

In [ ]:
df = pd.read_csv(CSV)
assert len(df) == 11098, len(df)
print(f"Corpus : {len(df)} segments, {df['usable'].sum()} évaluables")

In [ ]:
wav_files = list(CORPUS_DIR.rglob("*.wav"))
audio_map = {p.stem: p for p in wav_files}
assert len(audio_map) == len(wav_files), "collision de stem"
print(f"{len(audio_map)} fichiers audio")

In [ ]:
!nvidia-smi

## Section 1 — Travaux préliminaires

### 1.1 Diagnostic des modèles ⭐ 
Le vocabulaire CTC de MMS-wolof peut-il produire du français ? La fertilité du tokenizer Whisper sur le wolof ?

#### 1.1.1 Vocabulaire MMS (caractères français)
Vocabulaire MMS (caractères français)

In [ ]:
processor = AutoProcessor.from_pretrained("facebook/mms-1b-all")
processor.tokenizer.set_target_lang("wol")

vocab = set(processor.tokenizer.get_vocab())
requis_fr = set("abcdefghijklmnopqrstuvwxyzéèêëàâîïôöùûüç")
manquants = requis_fr - {c.lower() for c in vocab}

print(f"Vocabulaire wol : {len(vocab)} tokens")
print(f"Tokens : {sorted(vocab)}")
print(f"Caractères français absents : {sorted(manquants)}")

**MMS (adaptateur wolof)** — Le vocabulaire CTC compte 87 tokens et couvre
l'essentiel des caractères français (`c`, `v`, `é`, `è`, `ç`, `à`…) ; seuls
`ö` et `û`, marginaux en français, manquent. MMS-wolof n'est donc **pas**
plafonné au niveau des caractères : il peut structurellement produire du
français.

La limite est ailleurs. La tête CTC ayant été entraînée sur du texte wolof,
elle tendra à rendre les emprunts sous une forme wolofisée (`waksinasioŋ`
plutôt que `vaccination`) — un biais distributionnel que ce diagnostic ne
mesure pas, et que seule l'inférence révélera.

#### 1.1.2 Fertilité Whisper

In [ ]:
tok = WhisperTokenizer.from_pretrained("openai/whisper-small")

# mots wolof nettoyés, hors segments vides
mots = " ".join(df.loc[df["usable"], "text_clean"].dropna()).split()

n_tokens = sum(len(tok.encode(m, add_special_tokens=False)) for m in mots)
fertilite = n_tokens / len(mots)

print(f"Mots analysés     : {len(mots)}")
print(f"Fertilité Whisper : {fertilite:.2f} tokens/mot")

**Whisper** — La fertilité du tokenizer sur le wolof est de 1,88 token/mot,
comparable au français et à l'anglais (~1,3–2). Le BPE multilingue segmente
le wolof sans fragmentation excessive : pas de handicap de longueur de
séquence.

**Conclusion.** Aucun des deux modèles n'est écarté par un obstacle
structurel : MMS peut produire les caractères français, Whisper segmente
correctement le wolof. Les limites restantes — wolofisation des emprunts
(MMS et Whisper), alignement orthographique — sont distributionnelles et ne
se révéleront qu'à l'inférence. Les deux modèles justifient une baseline.

### 1.2 Inspection audio
Outil de debug : pour un segment, afficher côte à côte le texte (brut,
nettoyé, verdict), la forme d'onde, le mel-spectrogramme (l'entrée de
Whisper) et l'audio. Sert à vérifier une convention ou comprendre une erreur
ASR.

In [ ]:
from IPython.display import Audio, display

def inspecter_segment(seg, audio_map, sr=16000):
    audio, _ = librosa.load(audio_map[seg["file"]], sr=sr)
    clip = extract_segment(audio, sr, seg["start"], seg["end"])

    print("BRUT    :", seg["text"])
    print("NETTOYÉ :", seg["text_clean"])
    print("RAISON  :", seg["raison"], "| durée :", f"{seg['duration']:.2f}s")

    fig, ax = plt.subplots(2, 1, figsize=(12, 5))
    librosa.display.waveshow(clip, sr=sr, ax=ax[0])
    ax[0].set_ylabel("amplitude")

    mel = librosa.power_to_db(
        librosa.feature.melspectrogram(y=clip, sr=sr, n_mels=80),
        ref=np.max)
    img = librosa.display.specshow(mel, sr=sr, x_axis="time", y_axis="mel", ax=ax[1])
    fig.colorbar(img, ax=ax[1], format="%+2.0f dB")
    plt.tight_layout(); plt.show()

    display(Audio(clip, rate=sr))

In [ ]:
# un vide (backchannel) — dois entendre "mhm" ou un acquiescement
inspecter_segment(df[df["raison"]=="vide"].iloc[0], audio_map)

# un inintelligible — dois entendre du bruit ou de la parole noyée
inspecter_segment(df[df["raison"]=="inintelligible"].iloc[0], audio_map)

# un push message — dois entendre une annonce publicitaire propre
inspecter_segment(df[df["file"]=="wol_11490"].iloc[0], audio_map)


### 1.3 Métriques WER et CER
Calcul du WER (word error rate) et du CER (character error rate) via `jiwer`,
sur référence et hypothèse **toutes deux normalisées** par `normalize_for_wer`.
Le CER est essentiel ici : la segmentation en mots de Whisper est instable
(mots collés ou scindés), ce qui gonfle le WER sans refléter la qualité
d'audition. Le CER, insensible aux frontières de mots, corrige ce biais.

In [ ]:
def evaluer(reference, hypothese, **flags):
    """
    WER et CER entre une référence et une hypothèse.
    Les deux passent par normalize_for_wer (normalisation symétrique).
    flags : options passées à normalize_for_wer (split_hyphen, etc.)
    """
    ref = normalize_for_wer(reference, **flags)
    hyp = normalize_for_wer(hypothese, **flags)

    if not ref.strip():          # référence vide après normalisation
        return None

    return {
        "wer": jiwer.wer(ref, hyp),
        "cer": jiwer.cer(ref, hyp),
        "ref": ref,
        "hyp": hyp,
    }

In [ ]:
# cas parfait
print(evaluer("dama am ndox", "dama am ndox"))          # wer 0, cer 0

# une substitution sur trois mots
print(evaluer("dama am ndox", "dama am mbër"))          # wer 0.33

# le cas Whisper : mots collés — voir l'écart WER vs CER
print(evaluer("ñoo leen ko", "ñooleen ko"))            # wer haut, cer bas

## Section 2 — Baselines

Transcription par cinq modèles — Whisper vanilla, M9and2M/whisper-small-wolof,
MMS (adaptateur wol), CAYTU/whosper-large, soynade-research/Wolof-HuBERT-CTC —
sur un échantillon commun, puis mesure WER/CER via `evaluer`. Point
d'incertitude à vérifier en premier : Whisper (vanilla comme M9and2M et
Whosper, tous deux dérivés de son architecture) ne supporte pas nativement
le wolof comme langue cible — comportement à observer avant de forcer quoi
que ce soit. MMS et Wolof-HuBERT, tous deux en architecture CTC, n'ont pas
cette contrainte mais leurs propres limites (longueur minimale de segment) ;
Wolof-HuBERT se distingue par un continued pretraining spécifiquement wolof
(860h de spontané), contre l'adaptateur multilingue générique de MMS.

### 2.1 Les cinq modèles

#### 2.1.0 Briques communes

Le segment de référence utilisé pour calibrer chaque modèle avant tout
batch, et les deux fonctions de transcription (génératif Whisper, direct
CTC), réutilisées telles quelles dans toutes les sous-sections suivantes.

In [ ]:
# le segment de calibration

seg0 = df[df["text_clean"] == "li nga xamante ne bii moom nga yaakaaroon moom nga mën ci gis sa bopp"].iloc[0]
audio, _ = librosa.load(audio_map[seg0["file"]], sr=16000)
clip = extract_segment(audio, 16000, seg0["start"], seg0["end"])

In [ ]:
# transcrire, pour les modèles Whisper (génératifs) 

def transcrire(clip, sr, model, processor, language=None, max_new_tokens=200):
    inputs = processor(clip, sampling_rate=sr, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    kwargs = {
        "max_new_tokens": max_new_tokens,
        "no_repeat_ngram_size": 3,
    }
    if language:
        kwargs["language"] = language
        kwargs["task"] = "transcribe"
    with torch.no_grad():
        generated = model.generate(inputs["input_features"], **kwargs)
    return processor.batch_decode(generated, skip_special_tokens=True)[0]

In [ ]:
# transcrire_mms, pour les modèles CTC 
def transcrire_mms(clip, sr, model, processor):
    inputs = processor(clip, sampling_rate=sr, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    ids = torch.argmax(logits, dim=-1)
    return processor.batch_decode(ids)[0]

#### 2.1.1 Diagnostic Whisper vanilla

Whisper ne supporte pas le wolof dans son ensemble de langues d'entraînement
(`language="wo"` lève une erreur explicite — liste des langues supportées en
annexe si besoin). Trois configurations testées sur un même segment :

- **langue libre** → hallucination : une phrase française plausible mais sans
  rapport avec l'audio
- **français forcé** → résultat identique (confirme que Whisper avait déjà
  détecté le français par défaut)
- **arabe forcé** → effondrement en boucle de répétition

**Conclusion : Whisper-small vanilla est écarté de la baseline** — son WER
serait proche de 1.0 sur tout le corpus, sans valeur diagnostique. Il ne sera
pas évalué sur l'échantillon complet.

In [ ]:
processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model.eval()

In [ ]:
# Diagnostic : Whisper vanilla ne peut pas transcrire le wolof (jamais vu à l'entraînement)
processor_v = WhisperProcessor.from_pretrained("openai/whisper-small")
model_v = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model_v.eval()

print("VANILLA :", transcrire(clip, 16000, model_v, processor_v))
# → hallucination française, sans rapport avec l'audio (confirmé aussi en FR et AR forcés)

#### 2.1.2 Whisper fine-tuné (M9and2M/whisper-small-wolof)

Whisper-small fine-tuné sur une compilation wolof (ALFFA, WAXAL, FLEURS ;
57 h d'entraînement, clips < 6 s ; WER 0,17 annoncé sur son propre jeu de
test). Contrairement au vanilla, aucune langue n'est forcée : le modèle a
été spécifiquement adapté au wolof.

Sur un premier segment de KALLAAMA (radio spontanée, hors du domaine
d'entraînement de ce modèle), le résultat est qualitativement différent du
vanilla : le modèle produit du **vrai wolof**, avec des mots corrects
identifiables (`yaakaaroon`, `moom`), mais un WER élevé (0,87) — signe d'un
décalage de domaine plutôt que d'une incapacité structurelle. Le CER (0,49),
nettement plus bas, confirme qu'une partie de l'écart WER vient de la
segmentation en mots plutôt que d'erreurs acoustiques.

In [ ]:
processor_wo = WhisperProcessor.from_pretrained("M9and2M/whisper-small-wolof")
model_wo = WhisperForConditionalGeneration.from_pretrained("M9and2M/whisper-small-wolof")
model_wo = model_wo.to("cuda")
model_wo.eval()

In [ ]:
n_mots_ref = len(seg0["text_clean"].split())
max_tok = max(15, n_mots_ref * 2)

hyp = transcrire(clip, 16000, model_wo, processor_wo, max_new_tokens=max_tok)
print("RÉFÉRENCE :", seg0["text_clean"])
print("M9AND2M   :", hyp)
resultat = evaluer(seg0["text_clean"], hyp)
print(resultat)

#### 2.1.3 MMS (facebook/mms-1b-all, adaptateur wol)

Architecture CTC — fondamentalement différente de Whisper. Le diagnostic
de vocabulaire (section 1) a montré que l'adaptateur wolof couvre
l'essentiel des caractères français (seuls `ö` et `û` manquent), écartant
un plafond structurel au niveau caractère.

Contrairement à Whisper, un modèle CTC aligne l'audio directement sur des
caractères connus — il n'a pas de mécanisme génératif libre et ne peut
donc pas "halluciner" une phrase plausible mais sans rapport avec l'audio,
comme observé avec Whisper vanilla (section 4.1). Sur du wolof inconnu, un
échec attendu serait plutôt un enchaînement de caractères incohérent que
du texte fluide dans une autre langue — cette section vérifie si c'est
bien le cas.

Aucun réglage de génération n'est nécessaire ici (pas de `max_new_tokens`,
pas de risque de boucle) : la longueur de la sortie CTC est déterminée par
l'alignement avec l'audio, pas par une décision de génération token à
token.

In [ ]:
processor_mms = AutoProcessor.from_pretrained("facebook/mms-1b-all")
processor_mms.tokenizer.set_target_lang("wol")

model_mms = Wav2Vec2ForCTC.from_pretrained("facebook/mms-1b-all", target_lang="wol")
model_mms.load_adapter("wol")
model_mms = model_mms.to("cuda")
model_mms.eval()

In [ ]:
hyp_mms = transcrire_mms(clip, 16000, model_mms, processor_mms)
print("RÉFÉRENCE :", seg0["text_clean"])
print("MMS       :", hyp_mms)
print(evaluer(seg0["text_clean"], hyp_mms))

#### 2.1.4 Whosper-large (CAYTU/whosper-large)

Whisper-large-v2 fine-tuné pour le wolof avec un accent explicite sur le
code-switching wolof-français-anglais. Licence Apache-2.0. Testé sur les
mêmes échantillons que M9and2M et MMS pour une comparaison directe.

Modèle plus lourd (large-v2) — mêmes garde-fous de génération que pour
M9and2M (`no_repeat_ngram_size`, `max_new_tokens` proportionnel au nombre
de mots attendus), calibrés contre le même segment de référence avant de
lancer le batch.

In [ ]:
processor_wh = WhisperProcessor.from_pretrained("CAYTU/whosper-large")
model_wh = WhisperForConditionalGeneration.from_pretrained("CAYTU/whosper-large")
model_wh = model_wh.to("cuda")
model_wh.eval()

In [ ]:
n_mots_ref = len(seg0["text_clean"].split())
max_tok = max(15, n_mots_ref * 2)

hyp = transcrire(clip, 16000, model_wh, processor_wh, max_new_tokens=max_tok)
print("RÉFÉRENCE :", seg0["text_clean"])
print("WHOSPER   :", hyp)
resultat = evaluer(seg0["text_clean"], hyp)
print(resultat)

#### 2.1.5 Wolof-HuBERT (Soynade Research)

Architecture CTC (HuBERT), continued pretraining sur 860h de wolof
spontané collecté et filtré par Soynade Research — même équipe que le
benchmark Kallaama-Retrieval-Eval. Cinquième candidat.

Sur le segment de référence, résultat comparable au meilleur score du
jour (Whosper-large) : WER 0,27, CER 0,09 — les erreurs se concentrent
sur des mots grammaticaux courts, aucune dérive hors-langue, cohérent
avec l'architecture CTC. Aucune calibration de génération nécessaire
(contrairement aux modèles Whisper).

In [ ]:
processor_hb = AutoProcessor.from_pretrained("soynade-research/Wolof-HuBERT-CTC")
model_hb = AutoModelForCTC.from_pretrained("soynade-research/Wolof-HuBERT-CTC")
model_hb = model_hb.to("cuda")
model_hb.eval()

In [ ]:
hyp = transcrire_mms(clip, 16000, model_mms, processor_mms)
print("RÉFÉRENCE :", seg0["text_clean"])
print("HUBERT     :", hyp)
resultat = evaluer(seg0["text_clean"], hyp)
print(resultat)

### 2.2 Test elargis
Les quatre modèles utilisables (M9and2M, MMS, Whosper-large, Wolof-HuBERT)
évalués sur une base commune : trois strates KALLAAMA par durée, un
échantillon mixte élargi, un contrôle FLEURS, et un test qualitatif sur
audio personnel. Un seul modèle chargé en mémoire à la fois (GPU limité).

#### 2.2.0 Setup : Échantillons communs, fixés une fois
- ech_court / ech_moyen / ech_long (30 segments chacun, KALLAAMA)
- echantillon_large (~100 segments, mixte court/moyen/long/push/radio)
- fleurs_indices (15 exemples FLEURS)
- audio_perso (l'enregistrement spontané, chargé une fois)


In [ ]:
# Échantillons KALLAAMA
ech_court = df[df.usable & (df.duration < 2)].sample(30, random_state=2)
ech_moyen = df[df.usable & (df.duration >= 2) & (df.duration < 5)].sample(30, random_state=2)
ech_long  = df[df.usable & (df.duration >= 5) & (df.duration < 15)].sample(30, random_state=2)

echantillon_large = pd.concat([
    df[df.usable & (df.duration < 5) & df.has_codeswitch].sample(25, random_state=1),
    df[df.usable & (df.duration < 5) & ~df.has_codeswitch].sample(25, random_state=1),
    df[df.usable & (df.duration >= 5) & (df.duration < 15)].sample(25, random_state=1),
    df[df.usable & df["file"].str.startswith("wol_1")].sample(
        min(15, (df.usable & df["file"].str.startswith("wol_1")).sum()), random_state=1),
    df[df.usable & df["file"].str.startswith("wol_4")].sample(15, random_state=1),
]).drop_duplicates(subset=["file", "start", "end"])

# FLEURS
from datasets import load_dataset
fleurs_wo = load_dataset("google/fleurs", "wo_sn", split="test")

import random
random.seed(2)
fleurs_indices = random.sample(range(len(fleurs_wo)), 15)

# Fonctions d'évaluation par lot
def evaluer_strate(model, processor, echantillon, ctc=False):
    resultats = []
    for _, seg in echantillon.iterrows():
        audio, _ = librosa.load(audio_map[seg["file"]], sr=16000)
        clip = extract_segment(audio, 16000, seg["start"], seg["end"])
        if len(clip) < 400:
            continue
        if ctc:
            hyp = transcrire_mms(clip, 16000, model, processor)
        else:
            n_mots = len(seg["text_clean"].split())
            hyp = transcrire(clip, 16000, model, processor, max_new_tokens=max(15, n_mots*2))
        r = evaluer(seg["text_clean"], hyp)
        if r:
            resultats.append(r)
    df_res = pd.DataFrame(resultats)
    return df_res[["wer","cer"]].mean() if len(df_res) else None

def evaluer_fleurs(transcrire_fn, model, processor, indices, ctc=False):
    resultats = []
    for i in indices:
        ex = fleurs_wo[i]
        audio_f, sr_f = ex["audio"]["array"], ex["audio"]["sampling_rate"]
        ref_f = ex["transcription"]
        if ctc:
            hyp = transcrire_fn(audio_f, sr_f, model, processor)
        else:
            n_mots = len(ref_f.split())
            hyp = transcrire_fn(audio_f, sr_f, model, processor, max_new_tokens=max(15, n_mots*2))
        r = evaluer(ref_f, hyp)
        if r:
            resultats.append(r)
    df_res = pd.DataFrame(resultats)
    return df_res[["wer","cer"]].mean() if len(df_res) else None

# Dicts d'accumulation, remplis au fil de 5.2.1 à 5.2.4
resultats_matrice = {}
resultats_fleurs = {}

### 2.2.1 M9and2M
Charger → matrice (court/moyen/long/large) → FLEURS → audio perso → décharger

In [ ]:
processor_wo = WhisperProcessor.from_pretrained("M9and2M/whisper-small-wolof")
model_wo = WhisperForConditionalGeneration.from_pretrained("M9and2M/whisper-small-wolof").to("cuda")
model_wo.eval()

In [ ]:
if "audio_perso" in dir():
    hyp_perso = transcrire(audio_perso, sr_perso, model_wo, processor_wo, max_new_tokens=100)
    print("M9AND2M (perso) :", hyp_perso)
else:
    print("audio_perso pas encore chargé — à refaire plus tard")

In [ ]:

del model_wo, processor_wo
gc.collect(); torch.cuda.empty_cache()

### 2.2.2 MMS
Charger → matrice → FLEURS → audio perso → décharger

In [ ]:
processor_mms = AutoProcessor.from_pretrained("facebook/mms-1b-all")
processor_mms.tokenizer.set_target_lang("wol")
model_mms = Wav2Vec2ForCTC.from_pretrained("facebook/mms-1b-all", target_lang="wol").to("cuda")
model_mms.load_adapter("wol")
model_mms.eval()


In [ ]:
resultats_matrice["MMS"] = {
    "court": evaluer_strate(model_mms, processor_mms, ech_court, ctc=True),
    "moyen": evaluer_strate(model_mms, processor_mms, ech_moyen, ctc=True),
    "long":  evaluer_strate(model_mms, processor_mms, ech_long, ctc=True),
    "large": evaluer_strate(model_mms, processor_mms, echantillon_large, ctc=True),
}
print("MMS :", resultats_matrice["MMS"])

resultats_fleurs["MMS"] = evaluer_fleurs(transcrire_mms, model_mms, processor_mms, fleurs_indices, ctc=True)
print("MMS FLEURS :", resultats_fleurs["MMS"])

In [ ]:
if "audio_perso" in dir():
    hyp_perso = transcrire_mms(audio_perso, sr_perso, model_mms, processor_mms)
    print("MMS (perso) :", hyp_perso)
else:
    print("audio_perso pas encore chargé — à refaire plus tard")

In [ ]:
del model_mms, processor_mms
gc.collect(); torch.cuda.empty_cache()

### 2.2.3 Whosper-large
Charger → matrice → FLEURS → audio perso → décharger

In [ ]:
processor_wh = WhisperProcessor.from_pretrained("CAYTU/whosper-large")
model_wh = WhisperForConditionalGeneration.from_pretrained("CAYTU/whosper-large").to("cuda")
model_wh.eval()

In [ ]:
resultats_matrice["Whosper"] = {
    "court": evaluer_strate(model_wh, processor_wh, ech_court),
    "moyen": evaluer_strate(model_wh, processor_wh, ech_moyen),
    "long":  evaluer_strate(model_wh, processor_wh, ech_long),
    "large": evaluer_strate(model_wh, processor_wh, echantillon_large),
}
print("Whosper :", resultats_matrice["Whosper"])

resultats_fleurs["Whosper"] = evaluer_fleurs(transcrire, model_wh, processor_wh, fleurs_indices, ctc=False)
print("Whosper FLEURS :", resultats_fleurs["Whosper"])

In [ ]:
if "audio_perso" in dir():
    hyp_perso = transcrire_mms(audio_perso, sr_perso, model_mms, processor_mms)
    print("MMS (perso) :", hyp_perso)
else:
    print("audio_perso pas encore chargé — à refaire plus tard")

In [ ]:

del model_wh, processor_wh
gc.collect(); torch.cuda.empty_cache()

### 2.2.4 Wolof-HuBERT
Charger → matrice → FLEURS → audio perso → décharger

In [ ]:
processor_hb = AutoProcessor.from_pretrained("soynade-research/Wolof-HuBERT-CTC")
model_hb = AutoModelForCTC.from_pretrained("soynade-research/Wolof-HuBERT-CTC").to("cuda")
model_hb.eval()

In [ ]:
resultats_matrice["Wolof-HuBERT"] = {
    "court": evaluer_strate(model_hb, processor_hb, ech_court, ctc=True),
    "moyen": evaluer_strate(model_hb, processor_hb, ech_moyen, ctc=True),
    "long":  evaluer_strate(model_hb, processor_hb, ech_long, ctc=True),
    "large": evaluer_strate(model_hb, processor_hb, echantillon_large, ctc=True),
}
print("Wolof-HuBERT :", resultats_matrice["Wolof-HuBERT"])

resultats_fleurs["Wolof-HuBERT"] = evaluer_fleurs(transcrire_mms, model_hb, processor_hb, fleurs_indices, ctc=True)
print("Wolof-HuBERT FLEURS :", resultats_fleurs["Wolof-HuBERT"])

In [ ]:
if "audio_perso" in dir():
    hyp_perso = transcrire_mms(audio_perso, sr_perso, model_hb, processor_hb)
    print("HUBERT (perso) :", hyp_perso)
else:
    print("audio_perso pas encore chargé — à refaire plus tard")

In [ ]:
del model_hb, processor_hb
gc.collect(); torch.cuda.empty_cache()

### 2.2.5 Tableau récapitulatif
Assemblage de `resultats_matrice` (rempli au fil de 5.2.1 à 5.2.4) en un
DataFrame unique — la synthèse chiffrée de toute la section.

In [ ]:
matrice_df = pd.DataFrame({k: {s: (v[s]["wer"] if v[s] is not None else None) for s in v} 
                            for k, v in resultats_matrice.items()}).T
fleurs_df = pd.Series({k: (v["wer"] if v is not None else None) for k, v in resultats_fleurs.items()}, name="FLEURS")

tableau_final = matrice_df.join(fleurs_df)
print(tableau_final)

## Section 3 — Approfondissements


### 3.1 Chunking
Regrouper les courts, couper les longs. Mesurer l'écart de WER avec/sans.

### 3.2 Grille CS
WER eval_cs vs eval_mono, erreur sur les tokens :fra, WER autour des switches.

Note : la matrice élargie (5.2) mesure déjà un échantillon mixte
(`echantillon_large`) mais n'isole pas le WER spécifiquement CS vs mono
par modèle sur les échantillons fixes de 5.2.0 — le premier test du soir
(section 5.2, résultat non conservé dans le tableau final) avait montré
que cet écart s'estompait sur volume. À refaire ici, proprement, sur les
quatre modèles.

## Section 4 — Analyse d'erreurs & synthèse baseline

**Cinq modèles testés**
- Whisper-small vanilla — écarté d'entrée : wolof absent des langues
  supportées (`Unsupported language: wo`), hallucination hors-langue
  confirmée sur trois configurations (libre, français forcé, arabe forcé
  → boucle de répétition)
- M9and2M/whisper-small-wolof — Whisper fine-tuné sur une compilation
  wolof (ALFFA, WAXAL, FLEURS ; clips < 6 s), WER 0,17 annoncé sur son
  propre test
- MMS (facebook/mms-1b-all, adaptateur wol) — architecture CTC,
  multilingue générique (1000+ langues)
- CAYTU/whosper-large — Whisper-large-v2 + adaptateur LoRA, orienté
  code-switching wolof-français explicitement revendiqué
- soynade-research/Wolof-HuBERT-CTC — architecture CTC, continued
  pretraining spécifiquement wolof (860h de spontané)

**Résultats chiffrés** (WER, échantillons fixes communs à tous les modèles)

| | court | moyen | long | large (~100, mixte) | FLEURS |
|---|---:|---:|---:|---:|---:|
| M9and2M | 2,24 | 0,86 | 0,71 | 1,32 | 0,60 |
| MMS | 0,95 | 0,84 | 0,70 | 0,85 | 0,43 |
| **Whosper-large** | **0,50** | **0,56** | **0,24** | **0,43** | 0,47 |
| Wolof-HuBERT | 0,62 | 0,50 | 0,25 | 0,49 | 0,50 |

**Ce que la mesure établit**
1. Le WER décroît avec la longueur du segment, pour tous les modèles —
   confirmé de façon élargie et homogène (mêmes échantillons partout).
2. **Whosper-large est le meilleur candidat** sur 4 des 5 colonnes, avec
   un avantage marqué sur les segments courts (0,50 contre 0,62–2,24
   pour les autres) — précisément la strate où les modèles génératifs
   plus faibles (M9and2M) s'effondrent en confusion linguistique.
3. **Wolof-HuBERT est le concurrent le plus proche**, quasi ex æquo sur
   les segments longs (0,25 contre 0,24), et sans aucune calibration de
   génération nécessaire (architecture CTC, pas de risque de boucle ou
   de dérive de fin de séquence — contrairement aux trois modèles
   Whisper, qui ont tous demandé un réglage de `max_new_tokens` et
   `no_repeat_ngram_size`).
4. Le décalage de domaine (spontané KALLAAMA vs lecture FLEURS) reste
   visible pour M9and2M et MMS (WER nettement meilleur sur FLEURS que
   sur KALLAAMA), mais s'estompe pour Whosper-large et Wolof-HuBERT — les
   deux modèles les plus robustes semblent aussi les moins sensibles au
   registre.

**Limite connue de l'évaluation** : ~336 segments contiennent un nombre
verbalisé en français (`cent`, `vingt`...), dont 319 non détectés par
`is_usable` — même risque de désaccord de convention que les chiffres
numériques déjà exclus (notebook 1, section 5.4). Non corrigé dans cette
baseline ; les WER rapportés incluent ce biais résiduel.

**Ce que ça oriente pour la suite**
- **Whosper-large retenu comme candidat principal de warm-start**,
  Wolof-HuBERT comme alternative sérieuse (architecture plus simple à
  opérer, pas de garde-fou de génération à maintenir)
- Le point faible commun aux quatre modèles reste le wolof oral
  spontané pur sur segments courts — prioriser le fine-tuning sur ce
  registre précis
- Normaliser les nombres (chiffres et mots) dans `normalize_for_wer`
  plutôt que les exclure, pour réduire le biais résiduel identifié
- Test qualitatif sur audio personnel (locuteur natif, accent français,
  parole spontanée) en attente — à compléter une fois l'audio chargé,
  pour une contre-épreuve hors de toute distribution de corpus connue